In [1]:
# example of semi-supervised gan for mnist
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy import asarray
from numpy.random import randn
from numpy.random import randint
from tensorflow.keras.datasets.mnist import load_data
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Reshape
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Lambda
from tensorflow.keras.layers import Activation
from matplotlib import pyplot
from tensorflow.keras import backend
from tensorflow.keras import ops

# custom activation function
def custom_activation(output):
	logexpsum = backend.sum(backend.exp(output), axis=-1, keepdims=True)
	result = logexpsum / (logexpsum + 1.0)
	return result

# define the standalone supervised and unsupervised discriminator models
def define_discriminator(in_shape=(28,28,1), n_classes=10):
	# image input
	in_image = Input(shape=in_shape)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(in_image)
	fe = LeakyReLU(alpha=0.2)(fe)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	# flatten feature maps
	fe = Flatten()(fe)
	# dropout
	fe = Dropout(0.4)(fe)
	# output layer nodes
	fe = Dense(n_classes)(fe)
	# supervised output
	c_out_layer = Activation('softmax')(fe)
	# define and compile supervised discriminator model
	c_model = Model(in_image, c_out_layer)
	c_model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5), metrics=['accuracy'])
	# unsupervised output
	d_out_layer = Lambda(custom_activation)(fe)
	# define and compile unsupervised discriminator model
	d_model = Model(in_image, d_out_layer)
	d_model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))
	return d_model, c_model

# define the standalone generator model
def define_generator(latent_dim):
	# image generator input
	in_lat = Input(shape=(latent_dim,))
	# foundation for 7x7 image
	n_nodes = 128 * 7 * 7
	gen = Dense(n_nodes)(in_lat)
	gen = LeakyReLU(alpha=0.2)(gen)
	gen = Reshape((7, 7, 128))(gen)
	# upsample to 14x14
	gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)
	# upsample to 28x28
	gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)
	# output
	out_layer = Conv2D(1, (7,7), activation='tanh', padding='same')(gen)
	# define model
	model = Model(in_lat, out_layer)
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model):
	# make weights in the discriminator not trainable
	d_model.trainable = False
	# connect image output from generator as input to discriminator
	gan_output = d_model(g_model.output)
	# define gan model as taking noise and outputting a classification
	model = Model(g_model.input, gan_output)
	# compile model
	opt = Adam(learning_rate=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load the images
def load_real_samples():
	# load dataset
	(trainX, trainy), (_, _) = load_data()
	# expand to 3d, e.g. add channels
	X = expand_dims(trainX, axis=-1)
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	print(X.shape, trainy.shape)
	return [X, trainy]

# select a supervised subset of the dataset, ensures classes are balanced
def select_supervised_samples(dataset, n_samples=100, n_classes=10):
	X, y = dataset
	X_list, y_list = list(), list()
	n_per_class = int(n_samples / n_classes)
	for i in range(n_classes):
		# get all images for this class
		X_with_class = X[y == i]
		# choose random instances
		ix = randint(0, len(X_with_class), n_per_class)
		# add to list
		[X_list.append(X_with_class[j]) for j in ix]
		[y_list.append(i) for j in ix]
	return asarray(X_list), asarray(y_list)

# select real samples
def generate_real_samples(dataset, n_samples):
	# split into images and labels
	images, labels = dataset
	# choose random instances
	ix = randint(0, images.shape[0], n_samples)
	# select images and labels
	X, labels = images[ix], labels[ix]
	# generate class labels
	y = ones((n_samples, 1))
	return [X, labels], y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	z_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	z_input = z_input.reshape(n_samples, latent_dim)
	return z_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	z_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	images = generator.predict(z_input)
	# create class labels
	y = zeros((n_samples, 1))
	return images, y

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, c_model, latent_dim, dataset, n_samples=100):
	# prepare fake examples
	X, _ = generate_fake_samples(g_model, latent_dim, n_samples)
	# scale from [-1,1] to [0,1]
	X = (X + 1) / 2.0
	# plot images
	for i in range(100):
		# define subplot
		pyplot.subplot(10, 10, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(X[i, :, :, 0], cmap='gray_r')
	# save plot to file
	filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()
	# evaluate the classifier model
	X, y = dataset
	_, acc = c_model.evaluate(X, y, verbose=0)
	print('Classifier Accuracy: %.3f%%' % (acc * 100))
	# save the generator model
	filename2 = 'g_model_%04d.h5' % (step+1)
	g_model.save(filename2)
	# save the classifier model
	filename3 = 'c_model_%04d.h5' % (step+1)
	c_model.save(filename3)
	print('>Saved: %s, %s, and %s' % (filename1, filename2, filename3))

# train the generator and discriminator
def train(g_model, d_model, c_model, gan_model, dataset, latent_dim, n_epochs=20, n_batch=100):
	# select supervised dataset
	X_sup, y_sup = select_supervised_samples(dataset)
	print(X_sup.shape, y_sup.shape)
	# calculate the number of batches per training epoch
	bat_per_epo = int(dataset[0].shape[0] / n_batch)
	# calculate the number of training iterations
	n_steps = bat_per_epo * n_epochs
	# calculate the size of half a batch of samples
	half_batch = int(n_batch / 2)
	print('n_epochs=%d, n_batch=%d, 1/2=%d, b/e=%d, steps=%d' % (n_epochs, n_batch, half_batch, bat_per_epo, n_steps))
	# manually enumerate epochs
	for i in range(n_steps):
		# update supervised discriminator (c)
		[Xsup_real, ysup_real], _ = generate_real_samples([X_sup, y_sup], half_batch)
		c_loss, c_acc = c_model.train_on_batch(Xsup_real, ysup_real)
		# update unsupervised discriminator (d)
		d_model.trainable = True
		[X_real, _], y_real = generate_real_samples(dataset, half_batch)
		d_loss1 = d_model.train_on_batch(X_real, y_real)
		X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
		d_loss2 = d_model.train_on_batch(X_fake, y_fake)
		d_model.trainable = False
		# update generator (g)
		X_gan, y_gan = generate_latent_points(latent_dim, n_batch), ones((n_batch, 1))
		g_loss = gan_model.train_on_batch(X_gan, y_gan)
		# summarize loss on this batch
		print('>%d, c[%.3f,%.0f], d[%.3f,%.3f], g[%.3f]' % (i+1, c_loss, c_acc*100, d_loss1, d_loss2, g_loss))
		# evaluate the model performance every so often
		if (i+1) % (bat_per_epo * 1) == 0:
			summarize_performance(i, g_model, c_model, latent_dim, dataset)

# size of the latent space
latent_dim = 100
# create the discriminator models
d_model, c_model = define_discriminator()
# create the generator
g_model = define_generator(latent_dim)
# create the gan
gan_model = define_gan(g_model, d_model)
# load image data
dataset = load_real_samples()
# train model
train(g_model, d_model, c_model, gan_model, dataset, latent_dim)

       0/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28, 1) (60000,)
(100, 28, 28, 1) (100,)
n_epochs=20, n_batch=100, 1/2=50, b/e=600, steps=12000


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:83: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 930ms/step
>1, c[2.309,4], d[0.094,1.247], g[0.095]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
>2, c[2.306,7], d[0.861,1.246], g[0.095]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>3, c[2.312,7], d[1.014,1.245], g[0.095]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>4, c[2.310,7], d[1.079,1.244], g[0.095]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>5, c[2.311,7], d[1.115,1.244], g[0.095]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
>6, c[2.310,7], d[1.138,1.243], g[0.095]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
>7, c[2.312,7], d[1.154,1.242], g[0.096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>8, c[2.318,7], d[1.165,1.241], g[0.096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
>9, c[2.315,7], d[1.173,1.240], g[0.096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
>10, c[2.314,7], d[1.179,1.239], g[0.096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>11, c[2.319,8], d[1.183,1.238], g[0.096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>12, c[2.325,8], d[1.187,1.237], g[0.096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
>13, c[2.3

...

Classifier Accuracy: 10.138%
>Saved: generated_plot_8400.png, g_model_8400.h5, and c_model_8400.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
>8401, c[4.192,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
>8402, c[4.192,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>8403, c[4.193,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
>8404, c[4.193,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
>8405, c[4.193,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
>8406, c[4.194,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
>8407, c[4.194,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
>8408, c[4.195,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>8409, c[4.195,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
>8410, c[4.195,10], d[0.664,0.664], g[0.872]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
>8411, c[4.196,10], d[0.664,0.664

Classifier Accuracy: 10.092%
>Saved: generated_plot_9000.png, g_model_9000.h5, and c_model_9000.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>9001, c[4.255,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
>9002, c[4.255,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>9003, c[4.256,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
>9004, c[4.256,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
>9005, c[4.256,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>9006, c[4.256,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
>9007, c[4.256,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>9008, c[4.257,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
>9009, c[4.257,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>9010, c[4.257,10], d[0.664,0.664], g[0.874]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
>9011, c[4.257,10], d[0.664,0.664

Classifier Accuracy: 9.950%
>Saved: generated_plot_9600.png, g_model_9600.h5, and c_model_9600.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
>9601, c[4.184,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
>9602, c[4.184,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
>9603, c[4.185,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
>9604, c[4.185,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>9605, c[4.185,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
>9606, c[4.185,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
>9607, c[4.186,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
>9608, c[4.186,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
>9609, c[4.186,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>9610, c[4.186,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
>9611, c[4.186,10], d[0.663,0.663]

Classifier Accuracy: 9.655%
>Saved: generated_plot_10200.png, g_model_10200.h5, and c_model_10200.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
>10201, c[4.122,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
>10202, c[4.122,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
>10203, c[4.123,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
>10204, c[4.124,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>10205, c[4.124,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>10206, c[4.124,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
>10207, c[4.124,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
>10208, c[4.124,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>10209, c[4.125,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
>10210, c[4.126,10], d[0.663,0.663], g[0.875]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
>10211, c[4.126,10], 

: 

In [ ]:
# example of loading the classifier model and generating images
from numpy import expand_dims
from tensorflow.keras.models import load_model
from tensorflow.keras.datasets.mnist import load_data
# load the model
model = load_model('c_model_7200.h5')
# load the dataset
(trainX, trainy), (testX, testy) = load_data()
# expand to 3d, e.g. add channels
trainX = expand_dims(trainX, axis=-1)
testX = expand_dims(testX, axis=-1)
# convert from ints to floats
trainX = trainX.astype('float32')
testX = testX.astype('float32')
# scale from [0,255] to [-1,1]
trainX = (trainX - 127.5) / 127.5
testX = (testX - 127.5) / 127.5
# evaluate the model
_, train_acc = model.evaluate(trainX, trainy, verbose=0)
print('Train Accuracy: %.3f%%' % (train_acc * 100))
_, test_acc = model.evaluate(testX, testy, verbose=0)
print('Test Accuracy: %.3f%%' % (test_acc * 100))

Train Accuracy: 8.683%
Test Accuracy: 8.980%
